# VBZ Tramlinien — Übersicht & Streckenführung 2025

Vollständige Dokumentation aller VBZ Tramlinien mit Streckenführung beider Fahrtrichtungen, geordneten Haltestellen und interaktiver Karte — basierend auf GTFS-Daten 2025.

## Datenarchitektur

### Datenquellen

| Datenbedarf | Datei | Pfad | Jahr-Typ |
|:---|:---|:---|:---|
| Tramlinien (route_id, Liniennummer) | `gtfs_tram_routes.parquet` | `data/raw/gtfs/` | String `'2025'` |
| Trips: shape_id, direction_id, Endstation | `gtfs_tram_trips.parquet` | `data/raw/gtfs/` | String `'2025'` |
| Haltestellen-Reihenfolge | `gtfs_stop_times_2025.parquet` | `data/raw/gtfs/` | Int16 `2025` |
| Haltestellen-Koordinaten | `gtfs_tram_stops.parquet` | `data/raw/gtfs/` | String `'2025'` |
| Streckengeometrie (geglättet, ~300 Punkte/Linie) | `gtfs_tram_shapes.parquet` | `data/raw/gtfs/` | String `'2025'` |
| Offizielle Linienfarben | `LINE_COLORS` | `src/zh_tram_flow/config.py` | — |

> **Nicht verwenden:** `gtfs_tram_stop_times.parquet` (merged, alle Jahre) hat `stop_id`-Format `gen:23026:...` — inkompatibel mit `gtfs_tram_stops.parquet` (`ch:1:sloid:...`). Kein Join möglich.

### Herleitung: Linie → Strecke → Haltestellen

```
routes      year='2025'  →  17 Tramlinien (route_type=0, bereits vorgefiltert)
              ↓ route_id
trips       route_id  →  viele Trips pro Linie + Richtung
              direction_id: 0 / 1  (Int64, generisch — kein festes Hin/Rück)
              trip_headsign:  Endstation — einziger Linienname (route_long_name = NULL überall)
              shape_id:  Verweis auf geglättete Streckengeometrie
              ↓ repräsentativer Trip = Trip mit meisten Stops (Hauptstrecke, kein Kurzläufer)
stop_times  trip_id  →  stop_id sortiert nach stop_sequence  (year=2025 Int16)
              ↓ stop_id  (Format: ch:1:sloid:... — matcht gtfs_tram_stops ✓)
stops       stop_id  →  stop_name, stop_lat, stop_lon
shapes      shape_id  →  geglättete Koordinatenpunkte für die Kartenlinie
```

In [ ]:
import pandas as pd
import polars as pl
from IPython.display import display, HTML
import plotly.graph_objects as go

from zh_tram_flow.config import PATHS, LINE_COLORS, LINE_TEXT_COLORS

RAW_GTFS = PATHS['raw'] / 'gtfs'

# ── Tramlinien (17 Linien, route_type=0 bereits vorgefiltert, year='2025') ──
routes_df = (
    pl.scan_parquet(RAW_GTFS / 'gtfs_tram_routes.parquet')
    .filter(pl.col('year') == '2025')
    .select(['route_id', 'route_short_name'])
    .collect()
)

# ── Trips: direction_id ist Int64 (0 oder 1) ────────────────────────────────
trips_df = (
    pl.scan_parquet(RAW_GTFS / 'gtfs_tram_trips.parquet')
    .filter(pl.col('year') == '2025')
    .join(routes_df.lazy(), on='route_id')
    .select(['trip_id', 'route_id', 'route_short_name', 'direction_id', 'trip_headsign', 'shape_id'])
    .collect()
)

# ── Haltestellen-Koordinaten (year='2025', dedupliziert) ─────────────────────
stops_df = (
    pl.scan_parquet(RAW_GTFS / 'gtfs_tram_stops.parquet')
    .filter(pl.col('year') == '2025')
    .select(['stop_id', 'stop_name', 'stop_lat', 'stop_lon'])
    .collect()
    .unique(subset=['stop_id'])
)
stop_lookup = {r['stop_id']: r for r in stops_df.to_dicts()}

# ── Stop-Reihenfolge — nur Tram-Trips aus 6M-Zeilen-Parquet filtern ─────────
tram_trip_ids = trips_df['trip_id'].to_list()
stop_times_df = (
    pl.scan_parquet(RAW_GTFS / 'gtfs_stop_times_2025.parquet')
    .filter(pl.col('trip_id').is_in(tram_trip_ids))
    .select(['trip_id', 'stop_id', 'stop_sequence'])
    .sort(['trip_id', 'stop_sequence'])
    .collect()
)

# ── Repräsentativer Trip pro Linie+Richtung (meiste Stops = Hauptstrecke) ────
stop_counts = (
    stop_times_df
    .group_by('trip_id')
    .agg(pl.len().alias('n_stops'))
)
rep_trips = (
    trips_df
    .join(stop_counts, on='trip_id', how='left')
    .with_columns(pl.col('n_stops').fill_null(0))
    .sort('n_stops', descending=True)
    .unique(subset=['route_short_name', 'direction_id'], keep='first')
    .sort(['route_short_name', 'direction_id'])
)

# ── Streckengeometrie (geglättet, ~300 Punkte pro Shape) ─────────────────────
shape_ids = rep_trips['shape_id'].drop_nulls().to_list()
shapes_pd = (
    pl.scan_parquet(RAW_GTFS / 'gtfs_tram_shapes.parquet')
    .filter((pl.col('year') == '2025') & pl.col('shape_id').is_in(shape_ids))
    .select(['shape_id', 'shape_pt_lat', 'shape_pt_lon', 'shape_pt_sequence'])
    .sort(['shape_id', 'shape_pt_sequence'])
    .collect()
    .to_pandas()
)
shapes_dict = {}
for sid, grp in shapes_pd.groupby('shape_id', sort=True):
    shapes_dict[sid] = list(zip(grp['shape_pt_lat'], grp['shape_pt_lon']))

# ── Sortierte Linienliste ─────────────────────────────────────────────────────
tram_lines = sorted(
    rep_trips['route_short_name'].unique().to_list(),
    key=lambda x: int(x) if x.isdigit() else 999
)

print(f'✓ Daten geladen')
print(f'  Tramlinien:  {len(tram_lines)}')
print(f'  Trips:       {len(rep_trips)} repräsentative Trips (1 pro Linie+Richtung)')
print(f'  Stop-Times:  {len(stop_times_df):,} Zeilen')
print(f'  Shapes:      {len(shapes_dict)} Geometrien')

In [ ]:
fig = go.Figure()
for ln in reversed(tram_lines):
    color = LINE_COLORS.get(ln, '#999999')
    text_color = LINE_TEXT_COLORS.get(ln, '#FFFFFF')
    sub = rep_trips.filter(pl.col('route_short_name') == ln)
    dirs = {row['direction_id']: row['trip_headsign'] for row in sub.to_dicts()}
    dest_0 = (dirs.get(0) or '—').replace('Zürich, ', '')
    dest_1 = (dirs.get(1) or '—').replace('Zürich, ', '')
    label = f'{dest_1} ↔ {dest_0}'
    fig.add_trace(go.Bar(
        y=[f'L{ln}'], x=[1],
        orientation='h',
        marker_color=color,
        text=[label],
        textposition='inside',
        insidetextanchor='middle',
        hovertemplate=f'<b>Linie {ln}</b><br>{label}<extra></extra>',
        showlegend=False,
        textfont=dict(color=text_color, size=11),
    ))
fig.update_layout(
    title='VBZ Tramlinien 2025 — Offizielle Linienfarben & Endpunkte',
    xaxis=dict(showticklabels=False, showgrid=False, zeroline=False, range=[0, 1.05]),
    yaxis=dict(showgrid=False),
    height=600,
    barmode='stack',
    margin=dict(l=60, r=20, t=60, b=30),
    plot_bgcolor='white',
    paper_bgcolor='white',
)
fig.show()

In [ ]:
print('Inhaltsverzeichnis — VBZ Tramlinien 2025\n')
for ln in tram_lines:
    sub = rep_trips.filter(pl.col('route_short_name') == ln)
    dirs = {row['direction_id']: row['trip_headsign'] for row in sub.to_dicts()}
    dest_0 = (dirs.get(0) or '—').replace('Zürich, ', '')
    dest_1 = (dirs.get(1) or '—').replace('Zürich, ', '')
    print(f'  L{ln:2s}  {dest_1} ↔ {dest_0}')

In [ ]:
def get_line_stops(line_name: str, direction: int = 0) -> list[dict]:
    """Geordnete Haltestellen-Liste für eine Linie und Richtung (0 oder 1)."""
    sub = rep_trips.filter(
        (pl.col('route_short_name') == line_name) &
        (pl.col('direction_id') == direction)
    )
    if sub.height == 0:
        return []
    trip_id = sub.row(0, named=True)['trip_id']
    trip_st = stop_times_df.filter(pl.col('trip_id') == trip_id).sort('stop_sequence')
    result = []
    for row in trip_st.to_dicts():
        info = stop_lookup.get(row['stop_id'])
        if info:
            result.append({
                'seq': int(row['stop_sequence']),
                'name': str(info['stop_name']).replace('Zürich, ', ''),
                'lat': float(info['stop_lat']),
                'lon': float(info['stop_lon']),
            })
    return result


def plot_line_map(line_name: str) -> go.Figure:
    """Interaktive Karte: Streckenführung + Haltestellen in Linienfarbe."""
    color = LINE_COLORS.get(line_name, '#999999')
    fig = go.Figure()
    for direction in [0, 1]:
        sub = rep_trips.filter(
            (pl.col('route_short_name') == line_name) &
            (pl.col('direction_id') == direction)
        )
        if sub.height == 0:
            continue
        row = sub.row(0, named=True)
        headsign = (row.get('trip_headsign') or f'Richtung {direction}').replace('Zürich, ', '')
        stops = get_line_stops(line_name, direction)
        if not stops:
            continue
        is_main = (direction == 0)
        opacity = 1.0 if is_main else 0.6
        shape_id = row.get('shape_id')
        # Geglättete Streckenlinie via shape geometry
        if shape_id and shape_id in shapes_dict:
            pts = shapes_dict[shape_id]
            fig.add_trace(go.Scattermapbox(
                lat=[p[0] for p in pts],
                lon=[p[1] for p in pts],
                mode='lines',
                line=dict(width=4 if is_main else 2, color=color),
                opacity=opacity,
                hoverinfo='skip',
                showlegend=False,
            ))
        # Haltestellen als Punkte
        texts = [f'<b>{s["name"]}</b><br>Halt {s["seq"]}' for s in stops]
        fig.add_trace(go.Scattermapbox(
            lat=[s['lat'] for s in stops],
            lon=[s['lon'] for s in stops],
            mode='markers',
            marker=dict(size=9 if is_main else 6, color=color, opacity=opacity),
            text=texts,
            hovertemplate='%{text}<extra></extra>',
            name=f'→ {headsign}',
        ))
    # Karte auf Linienmitte zentrieren
    main_stops = get_line_stops(line_name, 0) or get_line_stops(line_name, 1)
    if main_stops:
        mid = main_stops[len(main_stops) // 2]
        center = dict(lat=mid['lat'], lon=mid['lon'])
    else:
        center = dict(lat=47.378, lon=8.540)
    fig.update_layout(
        mapbox=dict(style='carto-positron', center=center, zoom=12),
        margin=dict(l=0, r=0, t=40, b=0),
        height=500,
        title=f'Linie {line_name} — Streckenführung 2025',
        showlegend=True,
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.85)', borderwidth=1),
    )
    return fig


print('✓ get_line_stops() und plot_line_map() definiert')

In [ ]:
for ln in tram_lines:
    color = LINE_COLORS.get(ln, '#999999')
    sub = rep_trips.filter(pl.col('route_short_name') == ln)
    dirs = {row['direction_id']: row['trip_headsign'] for row in sub.to_dicts()}
    dest_0 = (dirs.get(0) or '—').replace('Zürich, ', '')
    dest_1 = (dirs.get(1) or '—').replace('Zürich, ', '')
    # Linien-Header
    style = f'color:{color}; border-left:6px solid {color}; padding-left:12px; margin-top:40px; font-family:sans-serif'
    display(HTML(f'<h2 style="{style}">Linie {ln} &nbsp;—&nbsp; {dest_1} ↔ {dest_0}</h2>'))
    # Haltestellen beider Richtungen
    for direction, dest in [(0, dest_0), (1, dest_1)]:
        stops = get_line_stops(ln, direction)
        if stops:
            print(f'  Richtung → {dest}  ({len(stops)} Halte)')
            for s in stops:
                print(f'    {s["seq"]:2d}. {s["name"]}')
        else:
            print(f'  Richtung {direction}: keine Daten')
        print()
    # Interaktive Karte
    fig = plot_line_map(ln)
    fig.show()
    print()
    print('─' * 60)
    print()